# 📏 Clase 6 — Métricas: ¿cómo saber si un modelo funciona?

Entrenaste un modelo. Hace predicciones. Pero **¿cómo sabés si es bueno?**
En esta clase aprendés las métricas de evaluación para regresión y clasificación,
y entendés el balance entre sesgo y varianza.

### ¿Qué vamos a cubrir?

| Sección | Concepto |
|---|---|
| 1 | Setup |
| 2 | ¿Qué es una métrica de evaluación? |
| 3 | Métricas de regresión: MAE, RMSE, R² |
| 4 | Métricas de clasificación: Accuracy, Precision, Recall, F1 |
| 5 | Sesgo vs varianza |
| 6 | Underfitting vs overfitting (demo visual) |
| 7 | Ejercicio evaluable |

> **Un modelo sin métricas es como un estudiante sin examen: no sabés si aprendió.**

In [ ]:
# ── 1. Setup ────────────────────────────────────────────────

import matplotlib.pyplot as plt
import random
import math

random.seed(42)

print("✅ Entorno listo.")

---
## 2. ¿Qué es una métrica de evaluación?

Una métrica es un **número que resume qué tan buenas son las predicciones** de un modelo.

### ¿Por qué importa?

- Sin métrica no podés comparar modelos
- Sin métrica no sabés si tu modelo mejoró o empeoró
- Sin métrica no podés decidir si tu modelo está listo para producción

### Dos tipos de problemas = dos familias de métricas

| Tipo de problema | Target | Predicción | Métricas |
|---|---|---|---|
| **Regresión** | Número continuo | 7.3, 4.1, 9.2... | MAE, RMSE, R² |
| **Clasificación** | Categoría | Spam/No-spam, 0/1 | Accuracy, Precision, Recall, F1 |

> 💡 **¿Cuándo usamos cada uno?**
> - Regresión: predecir un *valor* (precio, temperatura, nota)
> - Clasificación: predecir una *categoría* (spam, enfermedad, fraude)

---
## 3. Métricas de regresión: MAE, RMSE, R²

### Las 3 métricas fundamentales

| Métrica | Fórmula | ¿Qué mide? | Valor ideal |
|---|---|---|---|
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Error promedio absoluto | 0 |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | Error promedio, penaliza errores grandes | 0 |
| **R²** | $1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$ | % de varianza explicada por el modelo | 1.0 |

In [ ]:
# Ejemplo concreto: predicciones de notas

reales   = [7.0, 4.5, 8.0, 6.0, 9.5, 3.0, 5.5, 7.5, 8.5, 6.5]
modelo_A = [6.8, 5.0, 7.5, 6.2, 9.0, 3.5, 5.0, 7.0, 8.0, 6.8]  # modelo bueno
modelo_B = [5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0]  # baseline: siempre la media

print("📋 Datos del ejemplo:\n")
print(f"{'Real':<10} {'Modelo A':<12} {'Error A':<10} {'Modelo B':<12} {'Error B'}")
print("─" * 55)
for r, a, b in zip(reales, modelo_A, modelo_B):
    print(f"{r:<10} {a:<12} {abs(r-a):<10.1f} {b:<12} {abs(r-b):.1f}")

In [ ]:
# Implementación de las 3 métricas desde cero

def mae(reales, predichos):
    """Mean Absolute Error: error promedio absoluto."""
    n = len(reales)
    return sum(abs(r - p) for r, p in zip(reales, predichos)) / n

def rmse(reales, predichos):
    """Root Mean Squared Error: penaliza errores grandes."""
    n = len(reales)
    mse = sum((r - p) ** 2 for r, p in zip(reales, predichos)) / n
    return math.sqrt(mse)

def r_squared(reales, predichos):
    """R²: proporción de varianza explicada (1.0 = perfecto)."""
    media = sum(reales) / len(reales)
    ss_res = sum((r - p) ** 2 for r, p in zip(reales, predichos))  # error del modelo
    ss_tot = sum((r - media) ** 2 for r in reales)                  # varianza total
    return 1 - (ss_res / ss_tot) if ss_tot > 0 else 0

# Evaluar ambos modelos
print("📊 Comparación de modelos:\n")
print(f"{'Métrica':<10} {'Modelo A':<12} {'Modelo B (baseline)':<20} {'Mejor'}")
print("─" * 55)

metricas = [
    ("MAE",  mae(reales, modelo_A),   mae(reales, modelo_B)),
    ("RMSE", rmse(reales, modelo_A),  rmse(reales, modelo_B)),
    ("R²",   r_squared(reales, modelo_A), r_squared(reales, modelo_B)),
]

for nombre, va, vb in metricas:
    if nombre == "R²":
        mejor = "A ✅" if va > vb else "B"
    else:
        mejor = "A ✅" if va < vb else "B"
    print(f"{nombre:<10} {va:<12.4f} {vb:<20.4f} {mejor}")

print("\n📌 Modelo A gana en TODAS las métricas.")
print("   MAE/RMSE: más bajo = mejor.  R²: más alto = mejor.")

In [ ]:
# Visualización: real vs predicho

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Modelo A
ax1.scatter(reales, modelo_A, c="#3498db", s=100, edgecolors="#333", zorder=5)
ax1.plot([0, 10], [0, 10], "--", color="red", linewidth=1.5, label="Predicción perfecta")
ax1.set_xlabel("Nota real", fontsize=12)
ax1.set_ylabel("Nota predicha", fontsize=12)
ax1.set_title(f"Modelo A — MAE={mae(reales, modelo_A):.2f}, R²={r_squared(reales, modelo_A):.3f}",
              fontsize=11, fontweight="bold")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)

# Modelo B (baseline)
ax2.scatter(reales, modelo_B, c="#e74c3c", s=100, edgecolors="#333", zorder=5)
ax2.plot([0, 10], [0, 10], "--", color="red", linewidth=1.5, label="Predicción perfecta")
ax2.set_xlabel("Nota real", fontsize=12)
ax2.set_ylabel("Nota predicha", fontsize=12)
ax2.set_title(f"Modelo B (baseline) — MAE={mae(reales, modelo_B):.2f}, R²={r_squared(reales, modelo_B):.3f}",
              fontsize=11, fontweight="bold")
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)

fig.suptitle("Real vs Predicho: ¿cuánto se acercan los puntos a la diagonal?",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("📊 Cuanto más cerca de la diagonal roja, mejor el modelo.")
print("   Modelo A: puntos cerca de la línea. Modelo B: línea horizontal (siempre predice lo mismo).")

---
## 4. Métricas de clasificación: Accuracy, Precision, Recall, F1

Para problemas de clasificación (predecir una categoría), usamos métricas diferentes.

### La matriz de confusión

```
                    Predicho
                  Positivo  Negativo
Real  Positivo  │   TP    │   FN     │
      Negativo  │   FP    │   TN     │
```

| Sigla | Significado | Ejemplo (spam) |
|---|---|---|
| **TP** | True Positive | Era spam y dije spam ✅ |
| **TN** | True Negative | No era spam y dije no-spam ✅ |
| **FP** | False Positive | No era spam pero dije spam ❌ (email legítimo a spam) |
| **FN** | False Negative | Era spam pero dije no-spam ❌ (spam en inbox) |

In [ ]:
# Ejemplo: detector de spam
# 1 = spam, 0 = no-spam

y_real   = [1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0]
y_pred_A = [1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0]  # modelo A
y_pred_B = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]  # todo spam

def confusion_matrix(reales, predichos):
    """Calcular TP, TN, FP, FN."""
    tp = sum(1 for r, p in zip(reales, predichos) if r == 1 and p == 1)
    tn = sum(1 for r, p in zip(reales, predichos) if r == 0 and p == 0)
    fp = sum(1 for r, p in zip(reales, predichos) if r == 0 and p == 1)
    fn = sum(1 for r, p in zip(reales, predichos) if r == 1 and p == 0)
    return tp, tn, fp, fn

# Modelo A
tp_a, tn_a, fp_a, fn_a = confusion_matrix(y_real, y_pred_A)
print("📊 Modelo A — Matriz de confusión:")
print(f"  TP={tp_a}  FN={fn_a}")
print(f"  FP={fp_a}  TN={tn_a}")

# Modelo B
tp_b, tn_b, fp_b, fn_b = confusion_matrix(y_real, y_pred_B)
print(f"\n📊 Modelo B (siempre spam) — Matriz de confusión:")
print(f"  TP={tp_b}  FN={fn_b}")
print(f"  FP={fp_b}  TN={tn_b}")

In [ ]:
# Métricas de clasificación desde cero

def accuracy(tp, tn, fp, fn):
    """% de predicciones correctas."""
    return (tp + tn) / (tp + tn + fp + fn)

def precision(tp, fp):
    """De los que dije 'positivo', ¿cuántos realmente lo eran?"""
    return tp / (tp + fp) if (tp + fp) > 0 else 0

def recall(tp, fn):
    """De los positivos reales, ¿cuántos detecté?"""
    return tp / (tp + fn) if (tp + fn) > 0 else 0

def f1_score(prec, rec):
    """Media armónica de precision y recall."""
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

# Calcular para ambos modelos
modelos = [
    ("Modelo A", tp_a, tn_a, fp_a, fn_a),
    ("Modelo B (todo spam)", tp_b, tn_b, fp_b, fn_b),
]

print("📊 Comparación de métricas de clasificación:\n")
print(f"{'Métrica':<12} {'Modelo A':<12} {'Modelo B':<12} {'Mejor'}")
print("─" * 50)

for nombre_modelo, tp, tn, fp, fn in modelos:
    acc = accuracy(tp, tn, fp, fn)
    prec = precision(tp, fp)
    rec = recall(tp, fn)
    f1 = f1_score(prec, rec)

resultados = {}
for nombre_modelo, tp, tn, fp, fn in modelos:
    acc = accuracy(tp, tn, fp, fn)
    prec = precision(tp, fp)
    rec = recall(tp, fn)
    f1 = f1_score(prec, rec)
    resultados[nombre_modelo] = {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1}

for metrica in ["Accuracy", "Precision", "Recall", "F1"]:
    va = resultados["Modelo A"][metrica]
    vb = resultados["Modelo B (todo spam)"][metrica]
    mejor = "A ✅" if va >= vb else "B"
    print(f"{metrica:<12} {va:<12.3f} {vb:<12.3f} {mejor}")

print("\n📌 Modelo B tiene Recall=1.0 (detecta todo el spam) pero Precision=0.5 (marca todo como spam).")
print("   F1 es el balance entre ambos → Modelo A es mejor.")

In [ ]:
# ¿Cuándo importa más Precision vs Recall?

escenarios = [
    {
        "caso": "Filtro de spam",
        "prioridad": "PRECISION",
        "razon": "Un FP = email importante va a spam (peor que un spam en inbox)",
    },
    {
        "caso": "Diagnóstico de cáncer",
        "prioridad": "RECALL",
        "razon": "Un FN = no detectar un cáncer real (puede ser mortal)",
    },
    {
        "caso": "Detección de fraude",
        "prioridad": "RECALL",
        "razon": "Un FN = dejar pasar un fraude (pérdida económica)",
    },
    {
        "caso": "Recomendaciones de productos",
        "prioridad": "PRECISION",
        "razon": "Un FP = recomendar algo irrelevante (molesta al usuario)",
    },
    {
        "caso": "Moderación de contenido",
        "prioridad": "Depende",
        "razon": "FP = censurar algo legítimo. FN = dejar pasar algo ofensivo. Ambos son malos.",
    },
]

print("🎯 ¿Cuándo priorizar Precision vs Recall?\n")
for e in escenarios:
    emoji = "🎯" if e["prioridad"] == "PRECISION" else "🔍" if e["prioridad"] == "RECALL" else "⚖️"
    print(f"  {emoji} {e['caso']}: priorizar {e['prioridad']}")
    print(f"     Razón: {e['razon']}\n")

print("💡 No hay métrica universal. La mejor depende del COSTO de cada tipo de error.")

---
## 5. Sesgo vs varianza

Estos son los dos errores fundamentales de cualquier modelo de ML.

| Concepto | Definición | Síntomas |
|---|---|---|
| **Sesgo alto** (underfitting) | El modelo es demasiado simple para los datos | Error alto en train Y en test |
| **Varianza alta** (overfitting) | El modelo es demasiado complejo, memoriza el ruido | Error bajo en train PERO alto en test |
| **Balance ideal** | El modelo captura el patrón real sin memorizar ruido | Error bajo en train Y en test |

> 💡 **Analogía:** 
> - Sesgo alto = un estudiante que no estudió nada (no sabe los conceptos)
> - Varianza alta = un estudiante que se aprendió las respuestas de memoria (no entiende)
> - Balance = un estudiante que entendió los conceptos (puede responder preguntas nuevas)

In [ ]:
# Demo visual: sesgo vs varianza con regresión polinomial

# Datos reales: una parábola con ruido
random.seed(42)
x_puntos = [i * 0.5 for i in range(20)]
y_puntos = [0.3 * x**2 - 2 * x + 5 + random.gauss(0, 1.5) for x in x_puntos]

# Función para ajustar polinomio (implementación simple)
def ajustar_polinomio(x, y, grado):
    """Ajusta un polinomio por mínimos cuadrados."""
    n = len(x)
    # Construir la matriz de Vandermonde
    X = [[xi**p for p in range(grado + 1)] for xi in x]
    
    # Resolver X^T * X * c = X^T * y usando eliminación gaussiana
    # (versión simplificada)
    XTX = [[sum(X[k][i] * X[k][j] for k in range(n)) for j in range(grado + 1)] for i in range(grado + 1)]
    XTy = [sum(X[k][i] * y[k] for k in range(n)) for i in range(grado + 1)]
    
    # Resolución directa (Gauss-Jordan)
    m = grado + 1
    aug = [XTX[i][:] + [XTy[i]] for i in range(m)]
    
    for col in range(m):
        max_row = max(range(col, m), key=lambda r: abs(aug[r][col]))
        aug[col], aug[max_row] = aug[max_row], aug[col]
        pivot = aug[col][col]
        if abs(pivot) < 1e-12:
            continue
        aug[col] = [v / pivot for v in aug[col]]
        for row in range(m):
            if row != col:
                factor = aug[row][col]
                aug[row] = [aug[row][j] - factor * aug[col][j] for j in range(m + 1)]
    
    coefs = [aug[i][m] for i in range(m)]
    return coefs

def evaluar_polinomio(coefs, x):
    return sum(c * x**p for p, c in enumerate(coefs))

# Ajustar 3 modelos con diferente complejidad
grados = [1, 2, 15]  # lineal, cuadrático, polinomio de grado 15
nombres = ["Grado 1\n(UNDERFITTING)", "Grado 2\n(BALANCE)", "Grado 15\n(OVERFITTING)"]
colores = ["#e74c3c", "#2ecc71", "#3498db"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, grado, nombre, color in zip(axes, grados, nombres, colores):
    coefs = ajustar_polinomio(x_puntos, y_puntos, grado)
    
    # Curva ajustada
    x_curve = [i * 0.1 for i in range(100)]
    y_curve = [evaluar_polinomio(coefs, x) for x in x_curve]
    
    # Limitar valores extremos del overfitting
    y_curve = [max(-10, min(30, y)) for y in y_curve]
    
    ax.scatter(x_puntos, y_puntos, c="#333", s=50, zorder=5, label="Datos reales")
    ax.plot(x_curve, y_curve, color=color, linewidth=2, label=f"Polinomio grado {grado}")
    ax.set_title(nombre, fontsize=11, fontweight="bold")
    ax.set_ylim(-5, 25)
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=8)
    
    # MAE en train
    preds = [evaluar_polinomio(coefs, x) for x in x_puntos]
    mae_val = mae(y_puntos, preds)
    ax.text(0.05, 0.95, f"MAE={mae_val:.2f}", transform=ax.transAxes,
            fontsize=10, fontweight="bold", va="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))

fig.suptitle("Sesgo vs Varianza: el efecto de la complejidad del modelo",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("📊 Grado 1: demasiado simple → no captura la curva (UNDERFITTING)")
print("   Grado 2: justo → captura la tendencia sin memorizar ruido (BALANCE) ✅")
print("   Grado 15: exagerado → pasa por todos los puntos pero hace zigzag (OVERFITTING)")

---
## 6. Underfitting vs Overfitting: ¿cómo diagnosticar?

### La tabla de diagnóstico

| Error en Train | Error en Test | Diagnóstico | Solución |
|---|---|---|---|
| Alto | Alto | **Underfitting** (sesgo) | Modelo más complejo, más features |
| Bajo | Bajo | **Balance** ✅ | ¡Listo! |
| Bajo | Alto | **Overfitting** (varianza) | Más datos, regularizar, modelo más simple |
| Alto | Bajo | Raro / datos mal particionados | Revisar la partición |

In [ ]:
# Simulación de curvas de aprendizaje (train vs validation)

random.seed(42)

# Simulamos 3 escenarios
epocas = list(range(1, 101))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Escenario 1: Underfitting
train_under = [3.0 - 0.5 * (1 - math.exp(-e/20)) + random.gauss(0, 0.05) for e in epocas]
val_under   = [3.5 - 0.5 * (1 - math.exp(-e/20)) + random.gauss(0, 0.08) for e in epocas]

axes[0].plot(epocas, train_under, color="#3498db", linewidth=2, label="Train")
axes[0].plot(epocas, val_under, color="#e74c3c", linewidth=2, label="Validation")
axes[0].set_title("UNDERFITTING\n(sesgo alto)", fontsize=11, fontweight="bold", color="#e74c3c")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Error")
axes[0].legend()
axes[0].grid(True, alpha=0.2)
axes[0].set_ylim(0, 4)
axes[0].text(50, 3.8, "Ambos errores altos", ha="center", fontsize=10, fontweight="bold", color="#e74c3c")

# Escenario 2: Balance
train_bal  = [2.0 * math.exp(-e/15) + 0.3 + random.gauss(0, 0.03) for e in epocas]
val_bal    = [2.5 * math.exp(-e/15) + 0.5 + random.gauss(0, 0.05) for e in epocas]

axes[1].plot(epocas, train_bal, color="#3498db", linewidth=2, label="Train")
axes[1].plot(epocas, val_bal, color="#e74c3c", linewidth=2, label="Validation")
axes[1].set_title("BALANCE ✅\n(justo)", fontsize=11, fontweight="bold", color="#2ecc71")
axes[1].set_xlabel("Época")
axes[1].legend()
axes[1].grid(True, alpha=0.2)
axes[1].set_ylim(0, 4)
axes[1].text(50, 3.8, "Ambos errores bajos y cercanos", ha="center", fontsize=10, fontweight="bold", color="#2ecc71")

# Escenario 3: Overfitting
train_over = [2.0 * math.exp(-e/10) + 0.1 + random.gauss(0, 0.02) for e in epocas]
val_over   = [2.0 * math.exp(-e/10) + 0.5 + 0.015 * e + random.gauss(0, 0.05) for e in epocas]

axes[2].plot(epocas, train_over, color="#3498db", linewidth=2, label="Train")
axes[2].plot(epocas, val_over, color="#e74c3c", linewidth=2, label="Validation")
axes[2].set_title("OVERFITTING\n(varianza alta)", fontsize=11, fontweight="bold", color="#e74c3c")
axes[2].set_xlabel("Época")
axes[2].legend()
axes[2].grid(True, alpha=0.2)
axes[2].set_ylim(0, 4)
axes[2].text(50, 3.8, "Train baja, Validation sube", ha="center", fontsize=10, fontweight="bold", color="#e74c3c")

fig.suptitle("Curvas de aprendizaje: diagnosticando el modelo",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("📊 Estas curvas son tu herramienta principal de diagnóstico.")
print("   Siempre graficá train vs validation error para detectar problemas.")

---
## 7. Ejercicio evaluable

### Consigna

Calculá métricas e interpretá resultados de modelos.

### Criterio de aprobación

- ✅ Calculaste MAE, RMSE y R² correctamente para el caso dado
- ✅ Calculaste Accuracy, Precision, Recall y F1 para clasificación
- ✅ Diagnosticaste correctamente underfitting vs overfitting
- ✅ Reflexión sobre cuándo usar cada métrica

In [ ]:
# ══════════════════════════════════════════════════════════════
#  EJERCICIO EVALUABLE — Métricas de evaluación
# ══════════════════════════════════════════════════════════════

# ── Parte 1: métricas de regresión ──────────────────────────

y_real_reg  = [6.0, 8.5, 3.0, 7.0, 5.5, 9.0, 4.0, 7.5, 6.5, 8.0]
y_pred_reg  = [5.5, 8.0, 4.0, 6.5, 5.0, 8.5, 4.5, 7.0, 6.0, 7.5]

# TODO: calcular las 3 métricas
mi_mae  = 0  # TODO
mi_rmse = 0  # TODO
mi_r2   = 0  # TODO

print(f"Mi MAE:  {mi_mae:.4f}  (esperado: ~0.5500)")
print(f"Mi RMSE: {mi_rmse:.4f}  (esperado: ~0.6245)")
print(f"Mi R²:   {mi_r2:.4f}  (esperado: ~0.8957)")

In [ ]:
# ── Parte 2: métricas de clasificación ──────────────────────

y_real_cls = [1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0]
y_pred_cls = [1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0]

# TODO: calcular TP, TN, FP, FN
mi_tp = 0  # TODO
mi_tn = 0  # TODO 
mi_fp = 0  # TODO
mi_fn = 0  # TODO

# TODO: calcular métricas
mi_accuracy  = 0  # TODO
mi_precision = 0  # TODO
mi_recall    = 0  # TODO
mi_f1        = 0  # TODO

print(f"TP={mi_tp} TN={mi_tn} FP={mi_fp} FN={mi_fn}")
print(f"Accuracy:  {mi_accuracy:.3f}")
print(f"Precision: {mi_precision:.3f}")
print(f"Recall:    {mi_recall:.3f}")
print(f"F1:        {mi_f1:.3f}")

In [ ]:
# ── Parte 3: diagnóstico de sesgo/varianza ──────────────────

# Un modelo tiene estos resultados:
# - MAE en Train:      0.82
# - MAE en Validation:  3.45

mi_diagnostico = {
    "diagnostico":  "",  # TODO: "underfitting", "overfitting" o "balance"
    "razon":        "",  # TODO: ¿por qué llegaste a esa conclusión?
    "solucion":     "",  # TODO: ¿qué harías para mejorar?
}

vacias = [k for k, v in mi_diagnostico.items() if v.strip() == ""]
if vacias:
    print(f"⚠️  Faltan: {vacias}")
else:
    print("✅ Diagnóstico completado.")
    for k, v in mi_diagnostico.items():
        print(f"  {k}: {v}")
    print("\n🎉 Guardá este notebook y envialo como entrega de la Clase 6.")

---
## Resumen de la clase

| Concepto | Lo que aprendiste |
|---|---|
| MAE | Error promedio absoluto (más bajo = mejor) |
| RMSE | Error que penaliza errores grandes (más bajo = mejor) |
| R² | % de varianza explicada (más alto = mejor, ideal = 1.0) |
| Accuracy | % de predicciones correctas en clasificación |
| Precision | De los positivos predichos, ¿cuántos eran reales? |
| Recall | De los positivos reales, ¿cuántos detecté? |
| F1 | Balance entre precision y recall |
| Underfitting | Modelo muy simple → error alto en todo |
| Overfitting | Modelo muy complejo → memoriza entrenamiento |

### Próxima clase

En la **Clase 7** vamos a construir tu **primera red neuronal con PyTorch**: neurona, activaciones, MLP y loop de entrenamiento completo.

> 📌 **Recordá:** elegí la métrica según el costo de cada tipo de error, no por costumbre.